# Traveling Salesman Problem — Solution (ported from `Solution.cpp`)

Task (`task.md`):

1. Create a set of cities (points) with coordinates `x, y` on a plane with height `z`.
   Cost A→B = Euclidean distance, if a road exists. Scenarios:
   - **a.** all direct connections / ~80% of possible connections
   - **b.** symmetrical / asymmetrical (asymmetrical: uphill +10%, downhill −10%)
   - Coordinates randomly from `x, y ∈ <-100, 100>`, `z ∈ <0, 50>`.
2. Represent the map as a weighted (directed) graph (cities = nodes, roads = edges).
3. Solve TSP: start from a chosen city, visit every city exactly once, return to start, minimize cost.
   State = partial/full path from starting city. Search = state tree.
   - **a.** full search: BFS and DFS
   - **b.** greedy: NN and Dijkstra
   - **c.** A* with inadmissible / admissible heuristics
   - **d.** ACO
4. Test each algorithm in each scenario for `n = 5…20` cities: path cost, time, memory.

> All algorithms are implemented here: DFS/BFS (§3a–3b), NN + Dijkstra-greedy (§3c), admissible/inadmissible A* (§3d), ACO (§3e), with the §4 benchmark. (`Solution.cpp` still has only DFS; the rest are C++ stubs.)

## 1. Setup — data model

In [8]:
import functools
import gc
import math
import random
import time
import tracemalloc
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Tuple

# --- Random number generator (mirrors random_xy(-100, 100), random_z(0, 50) in Solution.cpp) ---
random_generator = random.Random()

@dataclass
class City:
    x: int
    y: int
    z: int

@dataclass
class AlgorithmStats:
    time_ns: int = 0
    memory_usage: int = 0  # measured peak Python bytes via tracemalloc (see measured_algorithm)
    total_cost: float = float("inf")
    best_path: List[int] = field(default_factory=list)

def format_bytes(byte_count) -> str:
    """Human-readable byte count, e.g. 3288 -> '3.2 KB', 101698736 -> '97.0 MB'."""
    if byte_count == "" or byte_count is None:
        return ""
    byte_count = float(byte_count)
    for unit in ("B", "KB", "MB", "GB"):
        if byte_count < 1024 or unit == "GB":
            return f"{byte_count:.1f} {unit}" if unit != "B" else f"{int(byte_count)} B"
        byte_count /= 1024

def printout_stats(title: str, stats: AlgorithmStats) -> None:
    print(f"{title} -\tTime = {stats.time_ns} ns\tMemory Usage = {format_bytes(stats.memory_usage)} "
          f"({stats.memory_usage} B)\t"
          f"Total Cost = {stats.total_cost}\nPath = {stats.best_path}")

class Graph:
    def __init__(self) -> None:
        self.cities: List[City] = []
        self.paths: Dict[Tuple[int, int], float] = {}  # (source_id, destination_id) -> cost; directed

    def path_exists(self, source_id: int, destination_id: int) -> bool:
        return (source_id, destination_id) in self.paths

# SolutionAlgorithm = Callable[[Graph], AlgorithmStats]  (mirrors C++ using alias)
SolutionAlgorithm = Callable[[Graph], AlgorithmStats]

def euclidean_distance(city_a: City, city_b: City) -> float:
    delta_x = city_a.x - city_b.x
    delta_y = city_a.y - city_b.y
    delta_z = city_a.z - city_b.z
    return math.sqrt(delta_x * delta_x + delta_y * delta_y + delta_z * delta_z)


def measured_algorithm(algorithm: SolutionAlgorithm) -> SolutionAlgorithm:
    """Attach the actual peak memory usage (tracemalloc) to an algorithm's stats.

    The algorithm runs once untraced for authoritative timing and result, then a
    second time under tracemalloc solely to observe peak Python-object allocation.
    """
    @functools.wraps(algorithm)
    def wrapper(graph: Graph, **kwargs) -> AlgorithmStats:
        stats = algorithm(graph, **kwargs)
        gc.collect()
        tracemalloc.stop()
        tracemalloc.start()
        try:
            algorithm(graph, **kwargs)
            _, peak_bytes = tracemalloc.get_traced_memory()
        finally:
            tracemalloc.stop()
        stats.memory_usage = int(peak_bytes)
        return stats
    return wrapper


## 2. Graph creation — scenarios

In [1]:
def create_graph(city_count: int, is_complete: bool, is_symmetrical: bool) -> Graph:
    graph = Graph()
    for _ in range(city_count):
        x = random_generator.randint(-100, 100)
        y = random_generator.randint(-100, 100)
        z = 0 if is_symmetrical else random_generator.randint(0, 50)
        graph.cities.append(City(x, y, z))

    for source_id in range(city_count):
        for destination_id in range(city_count):
            if source_id == destination_id or graph.path_exists(source_id, destination_id):
                continue
            source_city = graph.cities[source_id]
            destination_city = graph.cities[destination_id]
            distance = euclidean_distance(source_city, destination_city)
            if not is_symmetrical and source_city.z != destination_city.z:
                goes_uphill = source_city.z < destination_city.z  # source -> destination goes up
                if goes_uphill:
                    graph.paths[(source_id, destination_id)] = distance * 1.1
                    graph.paths[(destination_id, source_id)] = distance * 0.9
                else:
                    graph.paths[(source_id, destination_id)] = distance * 0.9
                    graph.paths[(destination_id, source_id)] = distance * 1.1
            else:
                graph.paths[(source_id, destination_id)] = distance
                graph.paths[(destination_id, source_id)] = distance

    if not is_complete:
        removed_roads: set[Tuple[int, int]] = set()
        roads_to_remove = int(0.2 * len(graph.paths))
        while len(removed_roads) < roads_to_remove:
            source_id = random_generator.randint(0, city_count - 1)
            destination_id = random_generator.randint(0, city_count - 1)
            if source_id == destination_id:
                continue
            removed_roads.add((source_id, destination_id))
            removed_roads.add((destination_id, source_id))
        for road in removed_roads:
            graph.paths.pop(road, None)
    return graph


## 3a. Full search — DFS.

In [2]:
@measured_algorithm
def dfs(graph: Graph) -> AlgorithmStats:
    city_count = len(graph.cities)
    assert city_count > 0
    stats = AlgorithmStats()

    best_cost = float("inf")
    max_path_depth = 0
    visited = [False] * city_count
    path: List[int] = []
    best_path: List[int] = []

    def visit(current_city: int, current_cost: float) -> None:
        nonlocal best_cost, max_path_depth, best_path
        max_path_depth = max(max_path_depth, len(path))
        # Prune (assumes non-negative edge costs).
        if current_cost >= best_cost:
            return
        if len(path) == city_count:
            return_cost = graph.paths.get((current_city, 0))
            if return_cost is not None:
                tour_cost = current_cost + return_cost
                if tour_cost < best_cost:
                    best_cost = tour_cost
                    best_path = list(path)
            return
        for next_city in range(city_count):
            if visited[next_city]:
                continue
            travel_cost = graph.paths.get((current_city, next_city))
            if travel_cost is None:
                continue
            visited[next_city] = True
            path.append(next_city)
            visit(next_city, current_cost + travel_cost)
            path.pop()
            visited[next_city] = False

    start_time_ns = time.perf_counter_ns()
    visited[0] = True
    path.append(0)
    visit(0, 0.0)
    end_time_ns = time.perf_counter_ns()
    stats.time_ns = end_time_ns - start_time_ns

    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats


## 3b. Full search — BFS

Exhaustive breadth-first search over the state tree (level-order expansion of partial paths from city 0).
Each state is `(current_city, visited_mask, path, current_cost)`; the queue starts with `(0, 1 << 0, [0], 0.0)` and expands all successors
one depth level at a time. Branch-and-bound pruning (`current_cost >= best_cost`) keeps optimality (edge costs are
non-negative). First solution found is NOT necessarily optimal (unlike DFS which dives deep immediately),
so the whole queue is drained (modulo pruning) and `best_cost` tracks the optimum — exactly like `dfs`,
but in BFS order. Memory usage is the actual peak Python-object allocation, measured with
`tracemalloc` by the `@measured_algorithm` wrapper (timing is taken from an untraced run).


In [3]:
from collections import deque

@measured_algorithm
def bfs(graph: Graph) -> AlgorithmStats:
    """Exhaustive BFS over the TSP state tree (level-order), with branch-and-bound pruning."""
    city_count = len(graph.cities)
    assert city_count > 0
    stats = AlgorithmStats()
    best_cost = float("inf")
    best_path: List[int] = []
    all_cities_visited_mask = (1 << city_count) - 1

    start_time_ns = time.perf_counter_ns()
    queue: deque = deque()
    queue.append((0, 1 << 0, [0], 0.0))  # (current_city, visited_mask, path, current_cost)
    while queue:
        current_city, visited_mask, path, current_cost = queue.popleft()
        if current_cost >= best_cost:
            continue  # prune (non-negative edge costs)
        if visited_mask == all_cities_visited_mask:
            return_cost = graph.paths.get((current_city, 0))
            if return_cost is not None:
                tour_cost = current_cost + return_cost
                if tour_cost < best_cost:
                    best_cost = tour_cost
                    best_path = list(path)
            continue
        for next_city in range(city_count):
            if visited_mask & (1 << next_city):
                continue
            travel_cost = graph.paths.get((current_city, next_city))
            if travel_cost is None:
                continue
            new_cost = current_cost + travel_cost
            if new_cost >= best_cost:
                continue
            queue.append((next_city, visited_mask | (1 << next_city), path + [next_city], new_cost))
    end_time_ns = time.perf_counter_ns()
    stats.time_ns = end_time_ns - start_time_ns

    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats


## 3c. Greedy search — Nearest Neighbour and Dijkstra-assisted greedy

- **Nearest Neighbour (NN):** from the current city always go to the cheapest directly-connected unvisited city.
  Fails (no tour) on incomplete graphs when the current city has no direct road to any unvisited city.
- **Dijkstra-greedy (`dijikstra`, correctly-spelled alias `dijkstra`):** same greedy idea, but the next city is the
  *nearest unvisited city by shortest-path distance* — each step runs Dijkstra from the current city and follows
  the shortest path to that target. Transit cities on the path count as visited, so this works on incomplete
  graphs by routing through intermediate cities. Note: `best_path` here is the full walk *including* transit
  revisits and the final return to 0 (length ≥ n+1), unlike the other methods which list n cities in visit order.


In [4]:
import heapq

@measured_algorithm
def nearest_neighbour(graph: Graph) -> AlgorithmStats:
    """Greedy NN: repeatedly visit the cheapest directly-reachable unvisited city."""
    city_count = len(graph.cities)
    assert city_count > 0
    stats = AlgorithmStats()
    start_time_ns = time.perf_counter_ns()
    visited = [False] * city_count
    path: List[int] = [0]
    visited[0] = True
    current_city = 0
    total_cost = 0.0
    tour_possible = True
    for _ in range(city_count - 1):
        best_next_city = None
        best_travel_cost = float("inf")
        for next_city in range(city_count):
            if visited[next_city]:
                continue
            travel_cost = graph.paths.get((current_city, next_city))
            if travel_cost is not None and travel_cost < best_travel_cost:
                best_travel_cost = travel_cost
                best_next_city = next_city
        if best_next_city is None:  # dead end (incomplete graph)
            tour_possible = False
            break
        visited[best_next_city] = True
        path.append(best_next_city)
        total_cost += best_travel_cost
        current_city = best_next_city
    if tour_possible:
        return_cost = graph.paths.get((current_city, 0))
        if return_cost is None:
            tour_possible = False
        else:
            total_cost += return_cost
    end_time_ns = time.perf_counter_ns()
    stats.time_ns = end_time_ns - start_time_ns
    if tour_possible:
        stats.total_cost = round(total_cost)
        stats.best_path = path
    return stats

def _dijkstra_shortest_paths(graph: Graph, source_city: int):
    """Standard Dijkstra from source_city over the road graph. Returns (distances, previous_city)."""
    city_count = len(graph.cities)
    distances = [float("inf")] * city_count
    previous_city = [-1] * city_count
    distances[source_city] = 0.0
    priority_queue = [(0.0, source_city)]
    settled = [False] * city_count
    while priority_queue:
        current_distance, city = heapq.heappop(priority_queue)
        if settled[city]:
            continue
        settled[city] = True
        for neighbour in range(city_count):
            travel_cost = graph.paths.get((city, neighbour))
            if travel_cost is None:
                continue
            new_distance = current_distance + travel_cost
            if new_distance < distances[neighbour] - 1e-12:
                distances[neighbour] = new_distance
                previous_city[neighbour] = city
                heapq.heappush(priority_queue, (new_distance, neighbour))
    return distances, previous_city

def _reconstruct_path(previous_city: List[int], source_city: int, destination_city: int) -> List[int]:
    path_segment: List[int] = []
    city = destination_city
    while city != -1:
        path_segment.append(city)
        if city == source_city:
            break
        city = previous_city[city]
    path_segment.reverse()
    return path_segment if path_segment and path_segment[0] == source_city else []

@measured_algorithm
def dijikstra(graph: Graph) -> AlgorithmStats:  # keep C++ typo in name
    """Greedy TSP via repeated Dijkstra (nearest unvisited city by shortest-path distance)."""
    city_count = len(graph.cities)
    assert city_count > 0
    stats = AlgorithmStats()
    start_time_ns = time.perf_counter_ns()
    visited = [False] * city_count
    visited[0] = True
    remaining_cities = city_count - 1
    walk: List[int] = [0]
    current_city = 0
    total_cost = 0.0
    tour_possible = True
    while remaining_cities > 0:
        distances, previous_city = _dijkstra_shortest_paths(graph, current_city)
        target_city = -1
        best_distance = float("inf")
        for candidate_city in range(city_count):
            if not visited[candidate_city] and distances[candidate_city] < best_distance:
                best_distance = distances[candidate_city]
                target_city = candidate_city
        if target_city == -1 or not math.isfinite(best_distance):
            tour_possible = False
            break
        path_segment = _reconstruct_path(previous_city, current_city, target_city)
        if not path_segment:
            tour_possible = False
            break
        for city in path_segment[1:]:
            travel_cost = graph.paths.get((walk[-1], city))
            if travel_cost is None:
                tour_possible = False
                break
            total_cost += travel_cost
            walk.append(city)
            if not visited[city]:
                visited[city] = True
                remaining_cities -= 1
        current_city = target_city
        if not tour_possible:
            break
    if tour_possible:
        distances, previous_city = _dijkstra_shortest_paths(graph, current_city)
        if not math.isfinite(distances[0]):
            tour_possible = False
        else:
            path_segment = _reconstruct_path(previous_city, current_city, 0)
            for city in path_segment[1:]:
                total_cost += graph.paths[(walk[-1], city)]
                walk.append(city)
    end_time_ns = time.perf_counter_ns()
    stats.time_ns = end_time_ns - start_time_ns
    if tour_possible:
        stats.total_cost = round(total_cost)
        stats.best_path = walk
    return stats

def dijkstra(graph: Graph) -> AlgorithmStats:
    return dijikstra(graph)  # correctly-spelled alias


## 3d. A* — admissible vs inadmissible heuristics

A* over the same state tree: priority `estimated_total = cost_so_far + heuristic`, where `cost_so_far` is the
cost of the path built so far and `heuristic` estimates the cost to finish the tour.
Closed set `best_known_cost[(current_city, visited_mask)]` prunes dominated states. The first full tour popped from the
heap is returned — optimal if the heuristic is admissible (never overestimates), suboptimal otherwise.

- **Admissible heuristic:** `MST(unvisited ∪ {current_city}) + min(return to 0)`.
  Any feasible completion must connect those nodes (≥ MST, using `min(i→j, j→i)` so asymmetry can't inflate it)
  and must return to 0 (≥ cheapest return edge). `inf` when disconnected. Never overestimates ⇒ optimal A*.
- **Inadmissible heuristic:** greedy NN completion cost from the state, inflated ×1.5 (a feasible completion is an
  *upper* bound on the optimum, so it overestimates ⇒ A* becomes a fast approximation). Dead ends fall back
  to `(remaining+1) × max_edge × 1.5`.


In [5]:
def _mst_cost(city_ids, graph: Graph) -> float:
    """Prim MST over `city_ids` with w(i,j) = min(cost(i->j), cost(j->i)); inf if disconnected."""
    city_ids = list(city_ids)
    if len(city_ids) <= 1:
        return 0.0
    cities_in_mst = {city_ids[0]}
    total_cost = 0.0
    cheapest_connection: Dict[int, float] = {}
    for city in city_ids[1:]:
        forward_cost = graph.paths.get((city_ids[0], city))
        backward_cost = graph.paths.get((city, city_ids[0]))
        possible_costs = [cost for cost in (forward_cost, backward_cost) if cost is not None]
        cheapest_connection[city] = min(possible_costs) if possible_costs else float("inf")
    while len(cities_in_mst) < len(city_ids):
        next_city = min((city for city in city_ids if city not in cities_in_mst),
                        key=lambda city: cheapest_connection[city], default=None)
        if next_city is None or not math.isfinite(cheapest_connection[next_city]):
            return float("inf")
        total_cost += cheapest_connection[next_city]
        cities_in_mst.add(next_city)
        for city in city_ids:
            if city in cities_in_mst:
                continue
            forward_cost = graph.paths.get((next_city, city))
            backward_cost = graph.paths.get((city, next_city))
            possible_costs = [cost for cost in (forward_cost, backward_cost) if cost is not None]
            if possible_costs:
                cheapest_connection[city] = min(cheapest_connection[city], min(possible_costs))
    return total_cost

def _h_admissible(graph: Graph, current_city: int, unvisited_cities) -> float:
    if not unvisited_cities:
        return graph.paths.get((current_city, 0), float("inf"))
    mst_cost = _mst_cost(set(unvisited_cities) | {current_city}, graph)
    if not math.isfinite(mst_cost):
        return float("inf")
    cheapest_return_cost = min((graph.paths.get((city, 0), float("inf")) for city in unvisited_cities),
                               default=float("inf"))
    return mst_cost + cheapest_return_cost

def _h_inadmissible(graph: Graph, current_city: int, unvisited_cities, max_edge_cost: float) -> float:
    if not unvisited_cities:
        return graph.paths.get((current_city, 0), float("inf"))
    remaining_cities = set(unvisited_cities)
    city = current_city
    path_cost = 0.0
    while remaining_cities:
        best_next_city = None
        best_edge_cost = float("inf")
        for candidate_city in remaining_cities:
            travel_cost = graph.paths.get((city, candidate_city))
            if travel_cost is not None and travel_cost < best_edge_cost:
                best_edge_cost = travel_cost
                best_next_city = candidate_city
        if best_next_city is None:  # dead end -> inflated fallback
            return (len(remaining_cities) + 1) * max_edge_cost * 1.5
        path_cost += best_edge_cost
        city = best_next_city
        remaining_cities.remove(best_next_city)
    return_cost = graph.paths.get((city, 0))
    if return_cost is None:
        return (len(unvisited_cities) + 1) * max_edge_cost * 1.5
    return (path_cost + return_cost) * 1.5

def _astar(graph: Graph, heuristic) -> AlgorithmStats:
    city_count = len(graph.cities)
    assert city_count > 0
    stats = AlgorithmStats()
    all_cities_visited_mask = (1 << city_count) - 1
    start_time_ns = time.perf_counter_ns()
    initial_heuristic = heuristic(graph, 0, {city for city in range(1, city_count)})
    priority_queue = [(initial_heuristic, 0.0, 0, 1 << 0, (0,))]  # (estimated_total, cost_so_far, current_city, visited_mask, path)
    best_known_cost: Dict[tuple, float] = {}
    best_cost = float("inf")
    best_path: List[int] = []
    while priority_queue:
        estimated_total, cost_so_far, current_city, visited_mask, path = heapq.heappop(priority_queue)
        if cost_so_far >= best_known_cost.get((current_city, visited_mask), float("inf")):
            continue
        best_known_cost[(current_city, visited_mask)] = cost_so_far
        if visited_mask == all_cities_visited_mask:
            return_cost = graph.paths.get((current_city, 0))
            if return_cost is not None:
                best_cost = cost_so_far + return_cost
                best_path = list(path)
                break  # first goal popped = A* answer (optimal only for admissible h)
            continue
        if estimated_total >= best_cost:
            continue
        unvisited_cities = {city for city in range(city_count) if not (visited_mask >> city & 1)}
        for next_city in unvisited_cities:
            travel_cost = graph.paths.get((current_city, next_city))
            if travel_cost is None:
                continue
            new_cost_so_far = cost_so_far + travel_cost
            if new_cost_so_far >= best_known_cost.get((next_city, visited_mask | (1 << next_city)), float("inf")):
                continue
            heuristic_value = heuristic(graph, next_city, unvisited_cities - {next_city})
            if not math.isfinite(heuristic_value):
                continue
            heapq.heappush(priority_queue,
                           (new_cost_so_far + heuristic_value, new_cost_so_far, next_city,
                            visited_mask | (1 << next_city), path + (next_city,)))
    end_time_ns = time.perf_counter_ns()
    stats.time_ns = end_time_ns - start_time_ns
    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats

@measured_algorithm
def admissible_astar(graph: Graph) -> AlgorithmStats:
    """A* with admissible MST heuristic (optimal, still exponential worst-case)."""
    return _astar(graph, _h_admissible)

@measured_algorithm
def inadmissible_astar(graph: Graph) -> AlgorithmStats:
    """A* with inadmissible inflated-NN heuristic (fast approximation)."""
    max_edge_cost = max(graph.paths.values()) if graph.paths else float("inf")
    return _astar(graph, lambda graph, current_city, unvisited_cities: _h_inadmissible(
        graph, current_city, unvisited_cities, max_edge_cost))


## 3e. Ant Colony Optimization (ACO)

Classic Dorigo ACO for TSP, adapted to directed/incomplete graphs: `ant_count` ants per iteration (default
`max(10, n)`, 100 iterations) start at city 0 and repeatedly pick the next unvisited city with probability
proportional to `pheromone^pheromone_weight * visibility^visibility_weight`
(`pheromone` is the learned trail strength, `visibility = 1/cost`). Only direct roads are candidates,
so ants can get stuck on incomplete graphs (that tour is discarded). After each iteration pheromones evaporate
(`× (1-evaporation_rate)`) and successful ants deposit `deposit_constant / tour_cost`. The best tour over all
iterations is returned. Defaults: `pheromone_weight=1.0, visibility_weight=3.0, evaporation_rate=0.3,
deposit_constant=100.0`. Pass `seed=` for reproducibility.


In [6]:
@measured_algorithm
def ant_colony(graph: Graph, ant_count=None, iteration_count: int = 100,
               pheromone_weight: float = 1.0, visibility_weight: float = 3.0,
               evaporation_rate: float = 0.3, deposit_constant: float = 100.0,
               seed=None) -> AlgorithmStats:
    city_count = len(graph.cities)
    assert city_count > 0
    stats = AlgorithmStats()
    local_random_generator = random.Random(seed)
    start_time_ns = time.perf_counter_ns()
    if city_count == 1:
        stats.time_ns = 0
        stats.total_cost = 0
        stats.best_path = [0]
        return stats
    pheromone = [[1.0] * city_count for _ in range(city_count)]
    best_cost = float("inf")
    best_path: List[int] = []
    ant_count = ant_count or max(10, city_count)
    for _ in range(iteration_count):
        completed_tours = []
        for _ in range(ant_count):
            current_city = 0
            visited = [False] * city_count
            visited[0] = True
            path = [0]
            tour_cost = 0.0
            tour_complete = True
            for _ in range(city_count - 1):
                feasible_cities = [city for city in range(city_count)
                                   if not visited[city] and (current_city, city) in graph.paths]
                if not feasible_cities:
                    tour_complete = False
                    break
                selection_weights = []
                total_weight = 0.0
                for city in feasible_cities:
                    travel_cost = graph.paths[(current_city, city)]
                    visibility = 1.0 / travel_cost if travel_cost > 0 else 1e9
                    selection_weight = (pheromone[current_city][city] ** pheromone_weight) * (
                        visibility ** visibility_weight)
                    selection_weights.append(selection_weight)
                    total_weight += selection_weight
                random_pick = local_random_generator.random() * total_weight
                cumulative_weight = 0.0
                chosen_city = feasible_cities[-1]
                for city, selection_weight in zip(feasible_cities, selection_weights):
                    cumulative_weight += selection_weight
                    if random_pick <= cumulative_weight:
                        chosen_city = city
                        break
                tour_cost += graph.paths[(current_city, chosen_city)]
                current_city = chosen_city
                visited[current_city] = True
                path.append(current_city)
            if tour_complete:
                return_cost = graph.paths.get((current_city, 0))
                if return_cost is None:
                    tour_complete = False
                else:
                    tour_cost += return_cost
            if tour_complete:
                completed_tours.append((tour_cost, path))
                if tour_cost < best_cost:
                    best_cost = tour_cost
                    best_path = list(path)
        for source_id in range(city_count):
            pheromone_row = pheromone[source_id]
            for destination_id in range(city_count):
                pheromone_row[destination_id] *= (1.0 - evaporation_rate)
        for tour_cost, path in completed_tours:
            pheromone_deposit = deposit_constant / tour_cost if tour_cost > 0 else deposit_constant
            previous_city = path[0]
            for city in path[1:]:
                pheromone[previous_city][city] += pheromone_deposit
                previous_city = city
            pheromone[previous_city][0] += pheromone_deposit
    end_time_ns = time.perf_counter_ns()
    stats.time_ns = end_time_ns - start_time_ns
    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats


## 4. Experiments — `n = 5…20`, 4 scenarios

Task §4: test each algorithm in each scenario (complete/incomplete × symmetrical/asymmetrical) for
`n = 5…20` cities: path cost, time, memory. Exact state-tree searches (`dfs`, `bfs`, `admissible_astar`)
are exponential, so they run only up to `MAX_EXACT_CITY_COUNT = 10` (larger `n` recorded as `SKIPPED`);
greedy / inadmissible-A* / ACO run the full range. `random_generator.seed(...)` keeps graphs reproducible.
Note `dijkstra`-greedy `best_path` is the full walk incl. transit revisits + final return, all others list
the n cities in visit order (cost always includes the return to 0). Every row is also saved to
`results.csv` (columns: `n, scenario, algorithm, status, total_cost, time_ns, memory_usage, memory_human, best_path`).
`memory_usage` holds exact peak bytes (numeric, for analysis); `memory_human` is the same value formatted
as KB/MB via `format_bytes` (same text shown in the printed `Memory Usage` field).


In [9]:
MAX_EXACT_CITY_COUNT = 10  # dfs / bfs / admissible A* only run up to this city count (exponential)
EXACT_ALGORITHM_NAMES = {"dfs", "bfs", "admissible_astar"}
ALGORITHMS: List[Tuple[str, SolutionAlgorithm]] = [
    ("dfs", dfs),
    ("bfs", bfs),
    ("nearest_neighbour", nearest_neighbour),
    ("dijkstra", dijkstra),
    ("inadmissible_astar", inadmissible_astar),
    ("admissible_astar", admissible_astar),
    ("ant_colony", ant_colony),
]
SCENARIOS = [
    ("complete symmetrical   ", True, True),
    ("complete asymmetrical ", True, False),
    ("incomplete symmetrical", False, True),
    ("incomplete asymmetrical", False, False),
]
RESULTS_CSV = "results.csv"

import csv

random_generator.seed(42)
result_rows = []
for city_count in range(5, 21):
    print(f"=== n={city_count} ===")
    for scenario_label, is_complete, is_symmetrical in SCENARIOS:
        graph = create_graph(city_count, is_complete, is_symmetrical)
        scenario = scenario_label.strip()
        for algorithm_name, algorithm in ALGORITHMS:
            if algorithm_name in EXACT_ALGORITHM_NAMES and city_count > MAX_EXACT_CITY_COUNT:
                print(f"{algorithm_name:20s} {scenario_label}: SKIPPED (n > {MAX_EXACT_CITY_COUNT})")
                result_rows.append({"n": city_count, "scenario": scenario, "algorithm": algorithm_name,
                                    "status": "skipped", "total_cost": "", "time_ns": "",
                                    "memory_usage": "", "memory_human": "", "best_path": ""})
                continue
            algorithm_kwargs = {"seed": 1} if algorithm_name == "ant_colony" else {}
            # The algorithm returns untraced timing/tour and its measured tracemalloc peak.
            stats = algorithm(graph, **algorithm_kwargs)  # type: ignore[call-arg]
            printout_stats(f"{algorithm_name:20s} {scenario_label}", stats)
            has_tour = math.isfinite(stats.total_cost)
            result_rows.append({"n": city_count, "scenario": scenario, "algorithm": algorithm_name,
                                "status": "ok" if has_tour else "no_tour",
                                "total_cost": stats.total_cost if has_tour else "",
                                "time_ns": stats.time_ns, "memory_usage": stats.memory_usage,
                                "memory_human": format_bytes(stats.memory_usage),
                                "best_path": "-".join(map(str, stats.best_path)) if has_tour else ""})
    print()

with open(RESULTS_CSV, "w", newline="") as results_file:
    writer = csv.DictWriter(results_file, fieldnames=["n", "scenario", "algorithm", "status",
                                                      "total_cost", "time_ns", "memory_usage", "memory_human",
                                                      "best_path"])
    writer.writeheader()
    writer.writerows(result_rows)
print(f"Saved {len(result_rows)} rows to {RESULTS_CSV}")


=== n=5 ===
dfs                  complete symmetrical    -	Time = 30738 ns	Memory Usage = 1.9 KB (1992 B)	Total Cost = 548
Path = [0, 3, 2, 1, 4]
bfs                  complete symmetrical    -	Time = 44695 ns	Memory Usage = 6.6 KB (6752 B)	Total Cost = 548
Path = [0, 3, 2, 1, 4]
nearest_neighbour    complete symmetrical    -	Time = 8236 ns	Memory Usage = 748 B (748 B)	Total Cost = 566
Path = [0, 4, 2, 3, 1]
dijkstra             complete symmetrical    -	Time = 49975 ns	Memory Usage = 1.9 KB (1936 B)	Total Cost = 566
Path = [0, 4, 2, 3, 1, 0]
inadmissible_astar   complete symmetrical    -	Time = 45476 ns	Memory Usage = 3.6 KB (3680 B)	Total Cost = 553
Path = [0, 1, 2, 3, 4]
admissible_astar     complete symmetrical    -	Time = 180152 ns	Memory Usage = 9.5 KB (9696 B)	Total Cost = 548
Path = [0, 3, 2, 1, 4]
ant_colony           complete symmetrical    -	Time = 6236712 ns	Memory Usage = 8.0 KB (8196 B)	Total Cost = 548
Path = [0, 4, 1, 2, 3]
dfs                  complete asymmetrical  -	T